# Pipeline QLoRA NER — DisTemIST con mistral-7b-instruct

Flujo completo de fine-tuning generativo para reconocimiento de enfermedades en textos clínicos en español.
Cubre configuración, entrenamiento con SFT, inferencia por oraciones y evaluación.

## Contenido

1. [Entorno y dependencias](#1-entorno-y-dependencias)
2. [Configuración](#2-configuracion)
3. [Carga del modelo](#3-carga-del-modelo)
4. [Adaptadores LoRA](#4-adaptadores-lora)
5. [Dataset](#5-dataset)
6. [Entrenamiento](#6-entrenamiento)
7. [Preparación para inferencia](#7-preparacion-para-inferencia)
8. [Inferencia](#8-inferencia)
9. [Evaluación](#9-evaluacion)

## 1. Entorno y dependencias

Instalación de paquetes e importación de librerías.

In [1]:
%%capture
!pip uninstall -y Pillow
!pip install "Pillow==11.3.0" --quiet

!pip install git+https://github.com/huggingface/transformers.git --upgrade --quiet

!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"

!pip install peft accelerate bitsandbytes trl==0.15.2 --quiet

!pip install -q spacy
!python -m spacy download es_core_news_md --quiet

## 2. Configuración

Hiperparámetros, rutas del dataset y credenciales.

In [ ]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# ── Modelo ────────────────────────────────────────────────────────────────────
MODEL_NAME     = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"

# ── LoRA ──────────────────────────────────────────────────────────────────────
LORA_R         = 64
LORA_ALPHA     = 128
LORA_DROPOUT   = 0
MAX_SEQ_LENGTH = 4096

# ── Entrenamiento ─────────────────────────────────────────────────────────────
BATCH_SIZE       = 1
GRAD_ACCUM       = 32
LEARNING_RATE    = 3e-5
NUM_TRAIN_EPOCHS = 2
WARMUP_RATIO     = 0.05
WEIGHT_DECAY     = 0.01
MAX_GRAD_NORM    = 0.3
TRAIN_SEED       = 3407
OUTPUT_DIR       = "outputs_lora"

# ── Dataset / chunking ────────────────────────────────────────────────────────
MAX_CHUNK_CHARS      = None   # None para oraciones sueltas
SPACY_MODEL          = "es_core_news_md"
MAX_NEGATIVE_RATIO   = None   # Ej: 0.5 => 1 negativo por cada 2 positivos
NEGATIVE_SAMPLE_SEED = 3407

# ── Prompt del sistema ────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "Actua como un sistema NER medico de alta precision.\n\n"
    "REGLAS DE EXTRACCION:\n"
    "1. Extrae exclusivamente entidades de tipo ENFERMEDAD.\n"
    "2. COPIA Y PEGA de forma literal: No cambies mayusculas, minusculas ni tildes.\n"
    "3. PROHIBIDO USAR SINONIMOS: Si el texto dice 'neoplasia', no escribas 'cancer'.\n"
    "4. REPETICIONES: Si una enfermedad aparece varias veces en el texto, debes listarla varias veces en lineas separadas.\n"
    "5. ORDEN: Extrae las menciones en el mismo orden en que aparecen en el texto.\n"
    "6. FORMATO: Únicamente el texto de la mención, una por línea. PROHIBIDO usar caracteres de lista al inicio (como '-', '*', '•', '1.').\n"
    "7. Si no hay nada, devuelve un texto vacio."
)

# ── Inferencia ────────────────────────────────────────────────────────────────
MAX_NEW_TOKENS   = 300
MAX_TEST_FILES   = 250   # None para procesar todos
PRINT_RAW_OUTPUT = True

# ── Rutas Distemist ───────────────────────────────────────────────────────────
PROJECT_ROOT         = "/kaggle/input/datasets/user"
DISTEMIST_ROOT       = f"{PROJECT_ROOT}/distemist/distemist"
TEXT_FILES_TRAIN_DIR = f"{DISTEMIST_ROOT}/text_files_train"
TEXT_FILES_TEST_DIR  = f"{DISTEMIST_ROOT}/text_files_test"
GS_TRAIN_TSV         = f"{DISTEMIST_ROOT}/distemist_subtrack1_training_mentions.tsv"
GS_TEST_TSV          = f"{DISTEMIST_ROOT}/distemist_subtrack1_test_mentions.tsv"
PREDICTIONS_TSV      = "distemist_llm_predictions.tsv"
EVAL_SUMMARY_JSON    = "evaluation_summary.json"

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None

print(f"Modelo base        : {MODEL_NAME}")
print(f"LoRA r={LORA_R}, alpha={LORA_ALPHA}")
print(f"MAX_SEQ_LENGTH     : {MAX_SEQ_LENGTH}")
print(f"MAX_CHUNK_CHARS    : {MAX_CHUNK_CHARS}")
print(f"Epochs             : {NUM_TRAIN_EPOCHS}")
print(f"MAX_NEW_TOKENS     : {MAX_NEW_TOKENS}")
print(f"MAX_TEST_FILES     : {MAX_TEST_FILES}")
print(f"MAX_NEGATIVE_RATIO : {MAX_NEGATIVE_RATIO}")

Modelo base        : unsloth/mistral-7b-instruct-v0.3-bnb-4bit
LoRA r=64, alpha=128
MAX_SEQ_LENGTH     : 4096
MAX_CHUNK_CHARS    : None
Epochs             : 2
MAX_NEW_TOKENS     : 300
MAX_TEST_FILES     : 250
MAX_NEGATIVE_RATIO : None


## 3. Carga del modelo

Carga cuantizada en 4-bit con distribución automática entre GPUs disponibles.

In [3]:
import torch
import unsloth
from unsloth import FastLanguageModel

print(f"GPUs disponibles: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {round(props.total_memory / 1024**3, 1)} GB VRAM")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.float16,
    load_in_4bit   = True,
    token          = hf_token,
)

if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer

print(f"\nModelo cargado: {MODEL_NAME}")
print(f"Parámetros totales: {sum(p.numel() for p in model.parameters()):,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
GPUs disponibles: 2
  GPU 0: Tesla T4 — 14.6 GB VRAM
  GPU 1: Tesla T4 — 14.6 GB VRAM
==((====))==  Unsloth 2026.6.1: Fast Mistral patching. Transformers: 5.10.0.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]


Modelo cargado: unsloth/mistral-7b-instruct-v0.3-bnb-4bit
Parámetros totales: 3,758,362,624


## 4. Adaptadores LoRA

Inyección de adaptadores LoRA sobre todas las proyecciones lineales del modelo.

In [4]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    target_modules = "all-linear",
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    task_type      = "CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Parámetros entrenables: {trainable:,}  ({100*trainable/total:.2f}%)")
model.print_trainable_parameters()

Parámetros entrenables: 167,772,160  (4.27%)
trainable params: 167,772,160 || all params: 7,415,795,712 || trainable%: 2.2624


## 5. Dataset

Segmentación por oraciones, construcción de prompts y ensamblado del dataset de entrenamiento.

In [5]:
import random
import numpy as np
import pandas as pd
import spacy
from pathlib import Path
from datasets import Dataset

nlp_spacy = spacy.load(SPACY_MODEL, disable=["ner", "lemmatizer"])
print(f"spaCy: {nlp_spacy.meta['name']} v{nlp_spacy.meta['version']}")


def chunk_text_by_sentences(text, nlp, max_chars=MAX_CHUNK_CHARS):
    """Divide el texto en chunks de oraciones completas con sus offsets.
    Si max_chars es None, devuelve una oración por chunk."""
    doc = nlp(text)
    if max_chars is None:
        return [(text[s.start_char:s.end_char], s.start_char, s.end_char) for s in doc.sents]

    chunks, current_chars, chunk_start, current_end = [], 0, None, None
    for sent in doc.sents:
        if current_chars + len(sent.text) > max_chars and chunk_start is not None:
            chunks.append((text[chunk_start:current_end], chunk_start, current_end))
            current_chars, chunk_start, current_end = 0, None, None
        if chunk_start is None:
            chunk_start = sent.start_char
        current_chars += len(sent.text)
        current_end = sent.end_char
    if chunk_start is not None:
        chunks.append((text[chunk_start:current_end], chunk_start, current_end))
    return chunks


def mentions_in_chunk(spans_with_offsets, chunk_start, chunk_end):
    return [span for span, off0, off1 in spans_with_offsets if chunk_start <= off0 < chunk_end]


def build_training_example(text, mentions):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Texto para analizar:\n{text}"},
        {"role": "assistant", "content": "\n".join(mentions) if mentions else ""},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


df_gs = pd.read_csv(GS_TRAIN_TSV, sep="\t").sort_values(["filename", "off0"])
text_dir = Path(TEXT_FILES_TRAIN_DIR)

positive_texts, negative_texts = [], []
for doc_id, group in df_gs.groupby("filename", sort=False):
    txt_path = text_dir / f"{doc_id}.txt"
    if not txt_path.exists():
        continue
    text = txt_path.read_text(encoding="utf-8")
    spans = list(zip(group["span"], group["off0"], group["off1"]))
    chunks = chunk_text_by_sentences(text, nlp_spacy)
    changed = True
    while changed:
        changed = False
        for i in range(len(chunks) - 1):
            _, cs, ce = chunks[i]
            _, ns, ne = chunks[i + 1]
            for _, off0, off1 in spans:
                if cs <= off0 < ce and off1 > ce:
                    chunks[i] = (text[cs:ne], cs, ne)
                    chunks.pop(i + 1)
                    changed = True
                    break
            if changed:
                break
    for chunk_text, char_start, char_end in chunks:
        mentions = mentions_in_chunk(spans, char_start, char_end)
        example  = build_training_example(chunk_text, mentions)
        (positive_texts if mentions else negative_texts).append(example)

if MAX_NEGATIVE_RATIO is None:
    negative_texts_used = negative_texts
else:
    max_neg = max(0, int(len(positive_texts) * MAX_NEGATIVE_RATIO))
    negative_texts_used = (
        random.Random(NEGATIVE_SAMPLE_SEED).sample(negative_texts, k=max_neg)
        if max_neg < len(negative_texts) else negative_texts
    )

formatted_texts = positive_texts + negative_texts_used
dataset = Dataset.from_dict({"text": formatted_texts})

lengths   = [len(tokenizer.encode(t)) for t in formatted_texts]
truncados = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)

print(f"Dataset: {len(dataset)} chunks (max_chars={MAX_CHUNK_CHARS})")
print(f"Positivos: {len(positive_texts)} | Negativos usados: {len(negative_texts_used)} / {len(negative_texts)} (ratio={MAX_NEGATIVE_RATIO})")
print(f"Tokens — min:{min(lengths)}  max:{max(lengths)}  media:{np.mean(lengths):.0f}  p95:{np.percentile(lengths,95):.0f}")
print(f"Truncados (>{MAX_SEQ_LENGTH}): {truncados} ({100*truncados/len(lengths):.1f}%)")
print(f"Menciones totales GS: {len(df_gs)}")

spaCy: core_news_md v3.8.0
Dataset: 11711 chunks (max_chars=None)
Positivos: 5185 | Negativos usados: 6526 / 6526 (ratio=None)
Tokens — min:14  max:412  media:68  p95:137
Truncados (>4096): 0 (0.0%)
Menciones totales GS: 8065


## 6. Entrenamiento

`SFTTrainer` con `DataCollatorForCompletionOnlyLM` para entrenar únicamente sobre las respuestas del asistente.

In [6]:
import gc
import torch
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"

tokenizer.padding_side = "right"

def filter_fn(examples):
    tokens = tokenizer(examples["text"], add_special_tokens=False)
    return [len(t) <= MAX_SEQ_LENGTH for t in tokens["input_ids"]]

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_SEQ_LENGTH)

train_data = (
    dataset
    .filter(filter_fn, batched=True, num_proc=2)
    .map(tokenize_fn, batched=True, num_proc=2, remove_columns=["text"])
)
print(f"Train: {len(train_data)} ejemplos")

response_template_ids = tokenizer.encode("[/INST]", add_special_tokens=False)
data_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=tokenizer,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_data,
    data_collator = data_collator,
    args = SFTConfig(
        max_seq_length              = MAX_SEQ_LENGTH,
        packing                     = False,
        dataset_kwargs              = {"skip_prepare_dataset": True},
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs            = NUM_TRAIN_EPOCHS,
        warmup_ratio                = WARMUP_RATIO,
        learning_rate               = LEARNING_RATE,
        fp16                        = True,
        bf16                        = False,
        logging_steps               = 10,
        optim                       = "paged_adamw_8bit",
        weight_decay                = WEIGHT_DECAY,
        max_grad_norm               = MAX_GRAD_NORM,
        lr_scheduler_type           = "linear",
        seed                        = TRAIN_SEED,
        output_dir                  = OUTPUT_DIR,
        report_to                   = "none",
        remove_unused_columns       = False,
        ddp_find_unused_parameters  = False,
        gradient_checkpointing      = True,
        eval_strategy               = "no",
        save_strategy               = "no",
    ),
)

trainer_stats = trainer.train()

for i in range(torch.cuda.device_count()):
    used  = round(torch.cuda.max_memory_reserved(i) / 1024**3, 3)
    total = round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 1)
    print(f"GPU {i}: VRAM pico = {used} GB / {total} GB ({100*used/total:.1f}%)")

print(f"Tiempo : {trainer_stats.metrics['train_runtime']:.0f} s  ({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"Loss   : {trainer_stats.metrics.get('train_loss', 'N/A'):.4f}")

Filter (num_proc=2):   0%|          | 0/11711 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/11711 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Train: 11711 ejemplos


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,711 | Num Epochs = 2 | Total steps = 732
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 32
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 32 x 1) = 32
 "-____-"     Trainable parameters = 167,772,160 of 7,415,795,712 (2.26% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,4.467029
20,0.969316
30,0.298941
40,0.306688
50,0.214845
60,0.229710
70,0.198998
80,0.148870
90,0.157887
100,0.136908


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!
GPU 0: VRAM pico = 5.869 GB / 14.6 GB (40.2%)
GPU 1: VRAM pico = 0.059 GB / 14.6 GB (0.4%)
Tiempo : 17813 s  (296.9 min)
Loss   : 0.1629


In [7]:
# ── Guardar adaptador LoRA ────────────────────────────────────────────────────
LORA_SAVE_DIR = "/kaggle/working/lora_adapter"
model.save_pretrained(LORA_SAVE_DIR)
tokenizer.save_pretrained(LORA_SAVE_DIR)
print(f"Adaptador guardado en {LORA_SAVE_DIR}")


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/lora_adapter/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in /kaggle/working/lora_adapter.


Adaptador guardado en /kaggle/working/lora_adapter


## 7. Preparación para inferencia

Liberación del trainer, configuración del pipeline generativo y listado de archivos de test.

In [8]:
import gc, time, re, json, logging
import pandas as pd
from pathlib import Path
from transformers import pipeline

LORA_SAVE_DIR = "/kaggle/working/lora_adapter"

if "trainer" in dir():
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
else:

    import torch
    from unsloth import FastLanguageModel
    from peft import PeftModel
    model_base, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype          = torch.float16,
        load_in_4bit   = True,
        token          = hf_token,
    )
    if hasattr(tokenizer, "tokenizer"):
        tokenizer = tokenizer.tokenizer
    model = PeftModel.from_pretrained(model_base, LORA_SAVE_DIR)
    print(f"Adaptador cargado desde {LORA_SAVE_DIR}")

logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)
model.eval()

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
)

def build_ner_messages(text):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Texto para analizar:\n{text}"},
    ]

txt_files = sorted(Path(TEXT_FILES_TEST_DIR).glob("*.txt"))
if MAX_TEST_FILES is not None:
    txt_files = txt_files[:MAX_TEST_FILES]

print(f"Archivos de test a procesar: {len(txt_files)}")

Archivos de test a procesar: 250


## 8. Inferencia

Segmentación por oraciones, extracción de menciones y ajuste de offsets al documento completo.

In [9]:
# ── Inferencia con checkpoint por archivo ─────────────────────────────────────
import json
from pathlib import Path

CHECKPOINT_FILE = "/kaggle/working/inference_checkpoint.json"

# Cargar checkpoint previo si existe
if Path(CHECKPOINT_FILE).exists():
    with open(CHECKPOINT_FILE) as f:
        ckpt = json.load(f)
    pred_rows       = ckpt["pred_rows"]
    inference_times = ckpt["inference_times"]
    done_files      = set(ckpt["done_files"])
    print(f"Retomando desde checkpoint: {len(done_files)} archivos ya procesados")
else:
    pred_rows, inference_times, done_files = [], [], set()

total_files = len(txt_files)

for i, txt_path in enumerate(txt_files):
    if txt_path.stem in done_files:
        print(f"[{i+1}/{total_files}] {txt_path.name} — ya procesado, saltando")
        continue

    text = txt_path.read_text(encoding="utf-8")
    raw_mentions = []
    t0 = time.time()

    chunks = chunk_text_by_sentences(text, nlp_spacy)

    for chunk_idx, (chunk_text, chunk_char_start, chunk_char_end) in enumerate(chunks):
        prompt = tokenizer.apply_chat_template(
            build_ner_messages(chunk_text),
            tokenize=False,
            add_generation_prompt=True,
        )
        try:
            result = pipe(
                prompt,
                max_new_tokens=MAX_NEW_TOKENS,
                max_length=None,
                do_sample=False,
            )[0]["generated_text"].strip()
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                gc.collect()
                torch.cuda.empty_cache()
            continue

        if PRINT_RAW_OUTPUT:
            print(f"  [chunk {chunk_idx+1}/{len(chunks)} | chars {chunk_char_start}-{chunk_char_end}]")
            print(f"  RAW: {repr(result)}")

        search_cursors = {}
        for mention in [line.strip() for line in result.splitlines() if line.strip()]:
            pattern = re.escape(mention)
            cursor  = search_cursors.get(mention, chunk_char_start)
            match   = re.search(pattern, text[cursor:chunk_char_end], re.IGNORECASE)
            if not match:
                match = re.search(pattern, text[chunk_char_start:chunk_char_end], re.IGNORECASE)
                if not match:
                    continue
                off0 = chunk_char_start + match.start()
                off1 = chunk_char_start + match.end()
            else:
                off0 = cursor + match.start()
                off1 = cursor + match.end()
            search_cursors[mention] = off1
            raw_mentions.append((text[off0:off1], off0, off1))

    t_infer = time.time() - t0
    inference_times.append(t_infer)

    raw_mentions.sort(key=lambda x: x[1])
    seen_offsets = set()
    deduped = []
    for span_text, off0, off1 in raw_mentions:
        if (off0, off1) not in seen_offsets:
            seen_offsets.add((off0, off1))
            deduped.append((span_text, off0, off1))

    for mark_idx, (span_text, off0, off1) in enumerate(deduped, 1):
        pred_rows.append({
            "filename": txt_path.stem,
            "mark":     f"T{mark_idx}",
            "label":    "ENFERMEDAD",
            "off0":     off0,
            "off1":     off1,
            "span":     span_text,
        })

    done_files.add(txt_path.stem)
    print(f"[{i+1}/{total_files}] {txt_path.name} | {t_infer:.2f}s | {len(chunks)} chunks | {len(deduped)} menciones")

    # Guardar checkpoint tras cada archivo
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"pred_rows": pred_rows, "inference_times": inference_times,
                   "done_files": list(done_files)}, f)

pd.DataFrame(pred_rows, columns=["filename", "mark", "label", "off0", "off1", "span"]).to_csv(
    PREDICTIONS_TSV, sep="\t", index=False
)
print(f"\nPredicciones guardadas en {PREDICTIONS_TSV}")
if inference_times:
    print(f"Tiempo medio por archivo: {sum(inference_times)/len(inference_times):.2f}s")


Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  [chunk 1/13 | chars 0-90]
  RAW: 'obesa\ndiabetes\nhipertensa'
  [chunk 2/13 | chars 91-170]
  RAW: 'neoplasia'
  [chunk 3/13 | chars 171-204]
  RAW: 'neoplasia'
  [chunk 4/13 | chars 205-347]
  RAW: 'neoplasia'
  [chunk 5/13 | chars 348-472]
  RAW: 'neoplasia'
  [chunk 6/13 | chars 473-667]
  RAW: 'tumor suprarrenal'
  [chunk 7/13 | chars 668-744]
  RAW: 'neoplasia'
  [chunk 8/13 | chars 744-799]
  RAW: 'neoplasia'
  [chunk 9/13 | chars 800-949]
  RAW: 'tumor localizado en la glándula suprarrenal'


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  [chunk 10/13 | chars 950-1003]
  RAW: 'neoplasia'
  [chunk 11/13 | chars 1004-1090]
  RAW: 'neoplasia'
  [chunk 12/13 | chars 1090-1294]
  RAW: 'neoplasia'
  [chunk 13/13 | chars 1295-1397]
  RAW: 'mielolipoma de la glándula adrenal'
[1/250] S0004-06142006000100010-1.txt | 12.92s | 13 chunks | 6 menciones
  [chunk 1/21 | chars 0-197]
  RAW: 'ENFERMEDAD\nadenocarcinoma gástrico\nhipertensión arterial'
  [chunk 2/21 | chars 198-412]
  RAW: 'anemia hipocrómica\nestreñimiento'
  [chunk 3/21 | chars 413-561]
  RAW: 'neoplasia'
  [chunk 4/21 | chars 561-806]
  RAW: 'adenocarcinoma gástrico\nanemia\nascitis por carcinomatosis\nobstrucción intestinal tumoral\ntumoración abdominal'
  [chunk 5/21 | chars 807-994]
  RAW: 'neoplasia'
  [chunk 6/21 | chars 995-1063]
  RAW: 'neoplasia'
  [chunk 7/21 | chars 1064-1164]
  RAW: 'pólipo inflamatorio en muñón gástrico\nrecidiva tumoral'
  [chunk 8/21 | chars 1165-1279]
  RAW: 'neoplasia'
  [chunk 9/21 | chars 1280-1460]
  RAW: 'tumoración quística'
  [

## 9. Evaluación

Métrica estricta por coincidencia exacta de offsets y métricas por solapamiento (IoU) a distintos umbrales.

In [10]:
import json
import pandas as pd


def prf(tp, fp, fn):
    p  = tp / (tp + fp) if (tp + fp) else 0.0
    r  = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f1


df_pred  = pd.read_csv(PREDICTIONS_TSV, sep="\t")
df_gs    = pd.read_csv(GS_TEST_TSV, sep="\t")
df_gs    = df_gs[df_gs["filename"].isin(df_pred["filename"].unique())]

set_gs   = set(zip(df_gs["filename"],   df_gs["label"],   df_gs["off0"],   df_gs["off1"]))
set_pred = set(zip(df_pred["filename"], df_pred["label"], df_pred["off0"], df_pred["off1"]))

tp, fp, fn = len(set_gs & set_pred), len(set_pred - set_gs), len(set_gs - set_pred)
p, r, f1   = prf(tp, fp, fn)

report = {"Estricta": {"tp": tp, "fp": fp, "fn": fn, "precision": round(p, 4), "recall": round(r, 4), "fscore": round(f1, 4)}}

print(f"── ESTRICTA ──  P={p:.4f}  R={r:.4f}  F1={f1:.4f}  (TP={tp} FP={fp} FN={fn})\n")

thresholds = [0.0, 0.5, 0.8]
results    = {t: {"tp": 0, "fp": 0, "fn": 0} for t in thresholds}

for filename in df_pred["filename"].unique():
    gs_ints   = list(zip(df_gs[df_gs["filename"] == filename]["off0"],   df_gs[df_gs["filename"] == filename]["off1"]))
    pred_ints = list(zip(df_pred[df_pred["filename"] == filename]["off0"], df_pred[df_pred["filename"] == filename]["off1"]))

    iou_matrix = sorted(
        [(max(0, min(p1,g1) - max(p0,g0)) / (max(p1,g1) - min(p0,g0)), pi, gi)
         for pi, (p0, p1) in enumerate(pred_ints)
         for gi, (g0, g1) in enumerate(gs_ints)
         if max(p1,g1) - min(p0,g0) > 0 and min(p1,g1) - max(p0,g0) > 0],
        reverse=True,
    )

    for t in thresholds:
        matched_p, matched_g = set(), set()
        for iou, pi, gi in iou_matrix:
            if iou >= t and pi not in matched_p and gi not in matched_g:
                matched_p.add(pi); matched_g.add(gi)
        tp_t = len(matched_p)
        results[t]["tp"] += tp_t
        results[t]["fp"] += len(pred_ints) - tp_t
        results[t]["fn"] += len(gs_ints)   - tp_t

print("── IoU ──")
for t in thresholds:
    tp_t, fp_t, fn_t = results[t]["tp"], results[t]["fp"], results[t]["fn"]
    p_t, r_t, f1_t   = prf(tp_t, fp_t, fn_t)
    report[f"IoU >= {t}"] = {"tp": tp_t, "fp": fp_t, "fn": fn_t,
                              "precision": round(p_t, 4), "recall": round(r_t, 4), "fscore": round(f1_t, 4)}
    print(f"IoU >= {t}:  P={p_t:.4f}  R={r_t:.4f}  F1={f1_t:.4f}  (TP={tp_t} FP={fp_t} FN={fn_t})")

with open(EVAL_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

── ESTRICTA ──  P=0.7663  R=0.7477  F1=0.7569  (TP=1941 FP=592 FN=655)

── IoU ──
IoU >= 0.0:  P=0.8934  R=0.8717  F1=0.8824  (TP=2263 FP=270 FN=333)
IoU >= 0.5:  P=0.8298  R=0.8097  F1=0.8197  (TP=2102 FP=431 FN=494)
IoU >= 0.8:  P=0.7773  R=0.7585  F1=0.7678  (TP=1969 FP=564 FN=627)
